# Canada Housing Data Cleaning

Combine the raw listing files for each province, clean them, and save one tidy dataset (`cleaned_canada.csv`) for analysis and the dashboard.

## 1. Load and combine the raw data

In [1]:
import pandas as pd
import numpy as np

In [2]:
data_ab = pd.read_csv('data_ab.csv', low_memory=False)
data_bc = pd.read_csv('data_bc.csv', low_memory=False)
data_mb = pd.read_csv('data_mb.csv', low_memory=False)
data_nb = pd.read_csv('data_nb.csv', low_memory=False)
data_nl = pd.read_csv('data_nl.csv', low_memory=False)
data_ns = pd.read_csv('data_ns.csv', low_memory=False)
data_nt = pd.read_csv('data_nt.csv', low_memory=False)
data_on = pd.read_csv('data_on.csv', low_memory=False)
data_pe = pd.read_csv('data_pe.csv', low_memory=False)
data_sk = pd.read_csv('data_sk.csv', low_memory=False)
data_yt = pd.read_csv('data_yt.csv', low_memory=False)

In [3]:
df = pd.concat([data_ab, data_bc, data_mb, data_nb, data_nl, 
                data_ns, data_nt, data_on, data_pe, data_sk,
                data_yt], axis=0) 
df = df.drop_duplicates()
df = df.T.drop_duplicates().T
df.shape

(98649, 32)

## 2. Drop rows missing key information

In [4]:
pd.set_option('display.max_columns', None)
df = df.dropna(subset=["streetAddress",
                       "addressLocality", "addressRegion", "price"])
df.shape

(97811, 32)

In [5]:
df.tail()

,streetAddress,addressLocality,addressRegion,postalCode,latitude,longitude,description,price,priceCurrency,property-beds,property-baths,property-sqft,Acreage,Air Conditioning,Basement,Bath,Exterior,Features,Fireplace,Garage,Heating,MLS® #,Property Tax,Property Type,Roof,Sewer,Square Footage,Subdivision,Waterfront,Parking,Flooring,Fireplace Features
134,"10307 104 Street, High Level, AB T0H1Z0",High Level,AB,T0H1Z0,58.51766,-117.143173,Welcome Home ! A wonderful home boasting SO M...,270000.0,CAD,4.0,2.0,"1,075",0.22,NaN,"Partially finished, Full",1.0,Vinyl siding,"Detached Garage, Parking Pad, Other",Yes,NaN,Forced air,A2153082,"$3,004 (2024)",Single Family,NaN,NaN,"1,075 SQFT",NaN,NaN,"Detached Garage, Parking Pad, Other","Laminate, Carpeted",NaN
143,"A & B - 29 KEG ROAD, Rainbow Lake, AB T0H2Y0",Rainbow Lake,AB,T0H2Y0,58.50808,-119.394859,Attention all investors. This is a unique oppo...,196999.0,CAD,6.0,4.0,NaN,NaN,NaN,"Finished, N/A",NaN,Wood,Attached Garage,NaN,NaN,"Baseboard heaters, Natural gas",X9014540,"$1,590",Single Family,NaN,Sanitary sewer,NaN,NaN,NaN,Attached Garage,NaN,NaN
182,"36 WHISKEY JACK CRESCENT, Fort Nelson, BC V0C1R0",Fort Nelson,BC,V0C1R0,58.865298,-122.84017,Check it out! Nearly 6 acres of mature trees o...,65000.0,CAD,NaN,0.0,NaN,5.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,R2739661,$242,Vacant Land,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
210,"157 FALCON DRIVE, Whitehorse, YT Y1A6G7",Whitehorse,YT,Y1A6G7,60.698432,-135.117177,"The custom-built 4-bedroom, 3-bathroom home yo...",749000.0,CAD,4.0,3.0,"2,576",0.19,NaN,NaN,NaN,NaN,"Wood, Electric, Conventional, Conventional",Yes,NaN,NaN,16178,"$3,439",Single Family,NaN,NaN,"2,576 SQFT",NaN,NaN,NaN,NaN,"Wood, Electric, Conventional, Conventional"
211,"A-28 IRON HORSE DRIVE, Whitehorse, YT Y1A6T4",Whitehorse,YT,Y1A6T4,60.693466,-135.109039,"This charming 2-bedroom, 2-bathroom duplex, bu...",465000.0,CAD,2.0,2.0,"1,380",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16147,"$2,200",Condo,NaN,NaN,"1,380 SQFT",NaN,NaN,NaN,NaN,NaN


## 3. Square footage and acreage

In [6]:
df[['Square Footage', 'property-sqft']].info() #both columns identical

<class 'pandas.core.frame.DataFrame'>
Index: 97811 entries, 7 to 211
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Square Footage  46732 non-null  object
 1   property-sqft   46489 non-null  object
dtypes: object(2)
memory usage: 2.2+ MB


In [7]:
df.loc[:, 'new-property-sqft'] = df['Square Footage'].str.replace(' SQFT', '', regex=False)
df.loc[:, 'new-property-sqft'] = df['new-property-sqft'].fillna(df['property-sqft'])
df = df.drop(columns=['Square Footage', 'property-sqft'])
df["Acreage"] = df["Acreage"].fillna(0)

/tmp/ipykernel_1134555/1192969766.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Acreage"] = df["Acreage"].fillna(0)


In [8]:
df.head()

,streetAddress,addressLocality,addressRegion,postalCode,latitude,longitude,description,price,priceCurrency,property-beds,property-baths,Acreage,Air Conditioning,Basement,Bath,Exterior,Features,Fireplace,Garage,Heating,MLS® #,Property Tax,Property Type,Roof,Sewer,Subdivision,Waterfront,Parking,Flooring,Fireplace Features,new-property-sqft
7,"1701 Coursier Avenue Unit# 1301, Revelstoke, B...",Revelstoke,BC,V0E2S3,50.976585,-118.173149,"Don't miss out on this rare 3 bedroom, 2 bath ...",839000.0,CAD,3.0,2.0,0.00,NaN,NaN,0.0,NaN,"Underground, Heated Garage",NaN,NaN,Heat Pump,10334677,"$3,478",Condo,NaN,Municipal sewage system,NaN,NaN,"Underground, Heated Garage","Tile, Carpeted, Vinyl",NaN,891
8,"12531 Lewis Bay, Boswell, BC V0B1A4",Boswell,BC,V0B1A4,49.47187,-116.770195,"Free living strata, no fees no stress. Your lo...",1150000.0,CAD,3.0,2.0,0.32,NaN,NaN,0.0,NaN,"Attached Garage, RV, See Remarks, Heated Garage",NaN,NaN,"Heat Pump, Baseboard heaters, Electric, See re...",10335620,"$3,500",Single Family,NaN,Septic tank,NaN,NaN,"Attached Garage, RV, See Remarks, Heated Garage",NaN,NaN,"1,881"
9,"2035 Boucherie Road Unit# 34, West Kelowna, BC...",West Kelowna,BC,V4T1Z6,49.82523,-119.603253,This cozy 2 bedroom home is located just steps...,149000.0,CAD,2.0,1.0,0.00,NaN,NaN,0.0,Aluminum,Aluminum,NaN,NaN,See remarks,10335525,$997,Single Family,"Tar & gravel, Unknown",Municipal sewage system,NaN,NaN,NaN,Laminate,NaN,912
10,"765 Westpoint Drive, Kelowna, BC V1W2Z4",Kelowna,BC,V1W2Z4,49.82186,-119.480143,"Imagine waking up to serene lake views, surrou...",1298000.0,CAD,5.0,4.0,0.69,NaN,NaN,1.0,NaN,Attached Garage,Yes,NaN,Forced air,10335485,"$6,995",Single Family,NaN,Municipal sewage system,NaN,NaN,Attached Garage,NaN,"Gas, Unknown","4,374"
11,"71 Spruce Avenue, Brooks, AB T1R0A5",Brooks,AB,T1R0A5,50.56341,-111.910375,Welcome to your perfect starter home in the hi...,285000.0,CAD,4.0,2.0,0.14,NaN,"Finished, Full",0.0,Aluminum siding,"Other, RV",NaN,NaN,"Forced air, High-Efficiency Furnace, Natural g...",A2194024,"$2,296 (2024)",Single Family,NaN,NaN,Sunnylea,NaN,"Other, RV","Carpeted, Other, Vinyl",NaN,NaN


## 4. Bathrooms check

In [9]:
# Get unique values from both columns
unique_values = pd.Index(df['property-baths'].dropna().unique()).union(df['Bath'].dropna().unique())

# Create a DataFrame with unique values as index
counts_df = pd.DataFrame(index=unique_values, columns=['property-baths', 'Bath'])

# Count occurrences and fill the DataFrame
counts_df['property-baths'] = df['property-baths'].value_counts().reindex(unique_values, fill_value=0)
counts_df['Bath'] = df['Bath'].value_counts().reindex(unique_values, fill_value=0)

counts_df

,property-baths,Bath
0.0,14428,7630
1.0,16867,17532
2.0,28083,1212
3.0,20532,59
4.0,11252,14
5.0,3020,1
6.0,1469,0
7.0,654,0
8.0,359,0
9.0,186,1


In [10]:
print(df[df['property-baths'] == 803]) #outlier/wrong row, drop
df[df['property-baths'] == 803] #outlier/wrong row, drop

                                   streetAddress addressLocality  \
9364  1214 Blackhead Road, St. John's, NL A1C5H2      St. John's   

     addressRegion postalCode   latitude  longitude  \
9364            NL     A1C5H2  47.510846 -52.673609   

                                            description     price  \
9364  This 2 bedroom home is located on Blackhead Ro...  198000.0   

     priceCurrency property-beds property-baths  Acreage Air Conditioning  \
9364           CAD           2.0          803.0      0.0              NaN   

     Basement   Bath          Exterior Features Fireplace Garage  \
9364       []  802.0  ["Vinyl siding"]       []        []    NaN   

                                Heating   MLS® #   Property Tax  \
9364  ["Baseboard heaters", "Electric"]  1280691  $1,323 (2024)   

      Property Type Roof            Sewer Subdivision Waterfront Parking  \
9364  Single Family   []  ["Septic tank"]         NaN        NaN      []   

                      Flooring Fi

,streetAddress,addressLocality,addressRegion,postalCode,latitude,longitude,description,price,priceCurrency,property-beds,property-baths,Acreage,Air Conditioning,Basement,Bath,Exterior,Features,Fireplace,Garage,Heating,MLS® #,Property Tax,Property Type,Roof,Sewer,Subdivision,Waterfront,Parking,Flooring,Fireplace Features,new-property-sqft
9364,"1214 Blackhead Road, St. John's, NL A1C5H2",St. John's,NL,A1C5H2,47.510846,-52.673609,This 2 bedroom home is located on Blackhead Ro...,198000.0,CAD,2.0,803.0,0.0,NaN,[],802.0,"[""Vinyl siding""]",[],[],NaN,"[""Baseboard heaters"", ""Electric""]",1280691,"$1,323 (2024)",Single Family,[],"[""Septic tank""]",NaN,NaN,[],"[""Hardwood"", ""Laminate""]",[],"1,592"


In [11]:
df = df[df['property-baths'] != 803] #drop the wrong row
df.shape

(97810, 31)

## 5. Remove duplicate listings

The same listing can show up in more than one province file, so I treat rows with the same address and price as duplicates.

In [12]:
df = df.drop_duplicates(subset=['streetAddress', 'price'])
df.shape

(87291, 31)

## 6. Convert numeric columns

After the transpose step every column is text (object), so convert the numeric ones back.

In [13]:
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['property-beds'] = pd.to_numeric(df['property-beds'], errors='coerce')
df['property-baths'] = pd.to_numeric(df['property-baths'], errors='coerce')
df['Acreage'] = pd.to_numeric(df['Acreage'], errors='coerce').fillna(0)
df['new-property-sqft'] = pd.to_numeric(df['new-property-sqft'].astype(str).str.replace(',', ''), errors='coerce')

df[['price', 'latitude', 'longitude', 'property-beds', 'property-baths', 'new-property-sqft']].isna().sum()

price                    0
latitude                 0
longitude                0
property-beds        11624
property-baths         672
new-property-sqft    45120
dtype: int64

In [14]:
# beds, baths and sqft are needed for the analysis, so drop rows without them
df = df.dropna(subset=['latitude', 'longitude', 'property-beds', 'property-baths', 'new-property-sqft'])
df = df[df['new-property-sqft'] > 0]
df.shape

(40796, 31)

## 7. Clean the text feature columns

Columns like Heating, Roof and Basement are messy free text, sometimes stored as lists (e.g. `["Forced Air", "Natural Gas"]`). First turn them into plain lowercase text.

In [15]:
def to_text(value):
    if pd.isna(value):
        return ''
    return str(value).lower().replace('[', '').replace(']', '').replace('"', '').strip()

text_cols = ['Garage', 'Parking', 'Basement', 'Exterior', 'Fireplace', 'Heating',
             'Flooring', 'Roof', 'Sewer', 'Waterfront']
for col in text_cols:
    df[col] = df[col].apply(to_text)

df['Heating'].value_counts().head(10)

Heating
forced air, natural gas        8255
forced air                     4493
baseboard, electric            3465
                               2402
electric                       1728
heat pump                      1438
baseboard                      1251
baseboard heaters, electric    1042
forced air, see remarks         772
radiant                         768
Name: count, dtype: int64

Then group them into a few simple categories using keywords. The first keyword found decides the category, so the order of the list matters.

In [16]:
def categorize(text, rules, default=np.nan):
    if text == '':
        return np.nan
    for label, keywords in rules.items():
        for keyword in keywords:
            if keyword in text:
                return label
    return default

basement_rules = {
    'Partial': ['partial', 'not full height'],
    'Finished': ['full', 'finished'],
    'No basement': ['none', 'crawl', 'slab', 'no basement'],
}
exterior_rules = {
    'Composite': ['composite'],
    'Vinyl': ['vinyl', 'siding'],
    'Brick': ['brick'],
    'Concrete': ['stucco', 'concrete', 'stone'],
    'Wood': ['wood', 'cedar'],
    'Metal': ['metal', 'aluminum', 'steel'],
}
heating_rules = {
    'forced air': ['forced air'],
    'boiler': ['boiler', 'hot water'],
    'heat pump': ['heat pump'],
    'radiant': ['baseboard', 'radiant', 'in floor', 'in-floor'],
    'alt heat': ['geo', 'solar', 'wood', 'pellet'],
    'fireplace': ['fireplace', 'stove'],
    'space heat': ['space heat'],
    'no heat': ['no heat'],
    'other': ['other'],
}
flooring_rules = {
    'carpet': ['carpet'],
    'wood': ['wood'],
    'tile': ['tile', 'ceramic', 'porcelain'],
    'vinyl': ['vinyl', 'linoleum'],
    'laminate': ['laminate'],
    'concrete': ['concrete'],
    'other': ['mixed', 'other'],
}
roof_rules = {
    'asphalt': ['asphalt'],
    'metal': ['metal', 'steel'],
    'fiberglass': ['fiberglass', 'fibreglass'],
    'cedar shake': ['cedar'],
    'shake': ['shake'],
    'tile': ['tile'],
    'clay': ['clay'],
    'tar': ['tar & gravel', 'tar and gravel', 'tar'],
    'flat': ['flat'],
    'shingle': ['shingle'],
}
sewer_rules = {
    'septic': ['septic'],
    'private': ['private'],
    'alternative': ['alternative', 'lagoon', 'holding', 'mound'],
    'municipal': ['municipal', 'sewer', 'sanitary'],
}

## 8. Build the final cleaned dataset

In [17]:
clean = pd.DataFrame({
    'City': df['addressLocality'],
    'Province': df['addressRegion'],
    'Latitude': df['latitude'],
    'Longitude': df['longitude'],
    'Price': df['price'],
    'Bedrooms': df['property-beds'],
    'Bathrooms': df['property-baths'],
    'Acreage': df['Acreage'],
    'Property Type': df['Property Type'],
    'Square Footage': df['new-property-sqft'],
})

# Yes/No features
clean['Garage'] = np.where((df['Garage'] == 'yes') | df['Parking'].str.contains('garage'), 'Yes', 'No')
clean['Parking'] = np.where(df['Parking'].str.contains('underground|parkade|carport|covered|visitor|stall'), 'Yes', 'No')

# Category features
clean['Basement'] = df['Basement'].apply(categorize, rules=basement_rules)
clean['Exterior'] = df['Exterior'].apply(categorize, rules=exterior_rules)

clean['Fireplace'] = np.where(df['Fireplace'].isin(['', '0', 'no', 'none']), 'No', 'Yes')

clean['Heating'] = df['Heating'].apply(categorize, rules=heating_rules)
clean['Flooring'] = df['Flooring'].apply(categorize, rules=flooring_rules)
clean['Roof'] = df['Roof'].apply(categorize, rules=roof_rules, default='other')
clean['Waterfront'] = np.where(df['Waterfront'] == 'yes', 'Yes', 'No')
clean['Sewer'] = df['Sewer'].apply(categorize, rules=sewer_rules, default='none').fillna('none')

# Pool, garden and balcony only appear inside the exterior/parking feature lists
features = df['Exterior'] + ' ' + df['Parking']
clean['Pool'] = np.where(features.str.contains('pool'), 'Yes', 'No')
clean['Garden'] = np.where(features.str.contains('garden|private yard'), 'Yes', 'No')
clean['Balcony'] = np.where(features.str.contains('balcony'), 'Yes', 'No')

clean = clean.reset_index(drop=True)
clean.shape

(40796, 23)

In [18]:
clean.head()

,City,Province,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Property Type,Square Footage,Garage,Parking,Basement,Exterior,Fireplace,Heating,Flooring,Roof,Waterfront,Sewer,Pool,Garden,Balcony
0,Revelstoke,BC,50.976585,-118.173149,839000.0,3.0,2.0,0.00,Condo,891.0,Yes,Yes,NaN,NaN,No,heat pump,carpet,NaN,No,municipal,No,No,No
1,Boswell,BC,49.471870,-116.770195,1150000.0,3.0,2.0,0.32,Single Family,1881.0,Yes,No,NaN,NaN,No,heat pump,NaN,NaN,No,septic,No,No,No
2,West Kelowna,BC,49.825230,-119.603253,149000.0,2.0,1.0,0.00,Single Family,912.0,No,No,NaN,Metal,No,NaN,laminate,tar,No,municipal,No,No,No
3,Kelowna,BC,49.821860,-119.480143,1298000.0,5.0,4.0,0.69,Single Family,4374.0,Yes,No,NaN,NaN,Yes,forced air,NaN,NaN,No,municipal,No,No,No
4,Maple Ridge,BC,49.221673,-122.596637,759900.0,3.0,2.0,0.00,Condo,1254.0,Yes,Yes,NaN,NaN,No,radiant,NaN,NaN,No,none,No,No,No


In [19]:
clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40796 entries, 0 to 40795
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   City            40796 non-null  object 
 1   Province        40796 non-null  object 
 2   Latitude        40796 non-null  float64
 3   Longitude       40796 non-null  float64
 4   Price           40796 non-null  float64
 5   Bedrooms        40796 non-null  float64
 6   Bathrooms       40796 non-null  float64
 7   Acreage         40796 non-null  float64
 8   Property Type   40796 non-null  object 
 9   Square Footage  40796 non-null  float64
 10  Garage          40796 non-null  object 
 11  Parking         40796 non-null  object 
 12  Basement        12273 non-null  object 
 13  Exterior        15058 non-null  object 
 14  Fireplace       40796 non-null  object 
 15  Heating         34967 non-null  object 
 16  Flooring        13470 non-null  object 
 17  Roof            8143 non-null  

In [20]:
clean.describe()

,Latitude,Longitude,Price,Bedrooms,Bathrooms,Acreage,Square Footage
count,40796.000000,40796.000000,4.079600e+04,40796.000000,40796.000000,40796.000000,40796.000000
mean,48.875452,-106.923825,1.116142e+06,3.206491,2.531915,2.429881,1813.070963
std,2.619857,22.804449,1.479178e+06,1.657250,1.453731,88.150516,2060.004880
min,42.045940,-135.856018,1.200000e+04,0.000000,0.000000,0.000000,1.000000
25%,48.448661,-122.922333,4.320000e+05,2.000000,2.000000,0.000000,972.000000
50%,49.201166,-120.241285,7.172500e+05,3.000000,2.000000,0.000000,1430.000000
75%,49.759504,-101.790635,1.289125e+06,4.000000,3.000000,0.180000,2237.000000
max,65.281488,-52.668600,5.880000e+07,37.000000,26.000000,8600.000000,290034.000000


In [21]:
for col in ['Property Type', 'Basement', 'Exterior', 'Heating', 'Flooring', 'Roof', 'Sewer']:
    print(clean[col].value_counts(dropna=False), '\n')

Property Type
Single Family        23753
Condo                11842
Townhome              2164
Condo/Townhome        1473
Duplex                 908
MultiFamily            254
Manufactured Home      210
Mobile Home            178
Vacant Land             14
Name: count, dtype: int64 

Basement
NaN            28523
Finished        7755
Partial         2286
No basement     2232
Name: count, dtype: int64 

Exterior
NaN          25738
Vinyl         9855
Brick         2054
Concrete      1683
Composite      714
Wood           641
Metal          111
Name: count, dtype: int64 

Heating
forced air    18440
radiant        8952
NaN            5829
heat pump      4106
boiler         2421
alt heat        562
other           402
no heat          46
fireplace        25
space heat       13
Name: count, dtype: int64 

Flooring
NaN         27326
carpet       5317
wood         3300
tile         1490
vinyl        1403
laminate     1148
other         781
concrete       31
Name: count, dtype: int64 

Roof
Na

## 9. Save

In [22]:
clean.to_csv('cleaned_canada.csv', index=False)